# Laboratorio Práctico: Proyecto Final
## Pregunta 1 — COVID-19

Código correspondiente a los ejercicios 1.1–1.10.

In [ ]:
# 1.1 — Obtener la página de Wikipedia mediante una solicitud HTTP
library(httr)
library(rvest)

get_wiki_covid19_page <- function() {
  wiki_base_url <- "https://en.wikipedia.org/w/index.php"
  url_param <- list(
    title = "Template:COVID-19_testing_by_country"
  )

  result <- GET(
    url = wiki_base_url,
    query = url_param
  )

  return(result)
}

result <- get_wiki_covid19_page()
result

In [ ]:
# 1.2 — Extraer la tabla de datos de pruebas de COVID-19
root <- read_html(result)

table_node <- html_nodes(root, "table")

covid_table <- html_table(table_node[2])

covid_data_frame <- as.data.frame(covid_table)

covid_data_frame

In [ ]:
# 1.3 — Preprocesar y exportar el marco de datos
preprocess_covid_data_frame <- function(data_frame) {

  data_frame <- data_frame[1:172, ]

  data_frame["Ref."] <- NULL
  data_frame["Units.b."] <- NULL

  names(data_frame) <- c(
    "country",
    "date",
    "tested",
    "confirmed",
    "confirmed.tested.ratio",
    "tested.population.ratio",
    "confirmed.population.ratio"
  )

  data_frame$country <- as.factor(data_frame$country)
  data_frame$date <- as.factor(data_frame$date)
  data_frame$tested <- as.numeric(gsub(",", "", data_frame$tested))
  data_frame$confirmed <- as.numeric(gsub(",", "", data_frame$confirmed))
  data_frame$confirmed.tested.ratio <- as.numeric(
    gsub(",", "", data_frame$confirmed.tested.ratio)
  )
  data_frame$tested.population.ratio <- as.numeric(
    gsub(",", "", data_frame$tested.population.ratio)
  )
  data_frame$confirmed.population.ratio <- as.numeric(
    gsub(",", "", data_frame$confirmed.population.ratio)
  )

  return(data_frame)
}

Covid19_frame <- preprocess_covid_data_frame(covid_data_frame)

Covid19_frame

write.table(
  Covid19_frame,
  "covid_data.csv",
  row.names = FALSE
)

In [ ]:
# 1.4 — Obtener un subconjunto del marco de datos
Covid19_data_frame_csv <- read.table(
  "covid_data.csv",
  header = TRUE,
  sep = ""
)

subset <- Covid19_data_frame_csv[
  1:10,
  c("country", "date", "confirmed")
]

subset

In [ ]:
# 1.5 — Calcular la relación de pruebas positivas de COVID-19 a nivel mundial
total_confirmed_cases <- sum(Covid19_data_frame_csv$confirmed)

total_tested_cases <- sum(Covid19_data_frame_csv$tested)

positive_ratio <- total_confirmed_cases / total_tested_cases

positive_ratio

In [ ]:
# 1.6 — Obtener una lista ordenada de nombres de países
country_list <- Covid19_data_frame_csv$country

country_list <- as.character(country_list)

A_to_Z <- sort(
  country_list,
  decreasing = FALSE
)

A_to_Z

In [ ]:
# 1.7 — Identificar países cuyo nombre comienza con United
uni_country <- grep(
  "United.+",
  Covid19_data_frame_csv$country,
  value = TRUE
)

print(uni_country)

In [ ]:
# 1.8 — Seleccionar dos filas por nombres de países y las tres columnas objetivo
country_a <- Covid19_data_frame_csv[
  Covid19_data_frame_csv$country == "Nigeria",
  c("country", "confirmed", "confirmed.population.ratio")
]

country_b <- Covid19_data_frame_csv[
  Covid19_data_frame_csv$country == "Ivory Coast",
  c("country", "confirmed", "confirmed.population.ratio")
]

country_a
country_b

In [ ]:
# 1.9 — Comparar la relación de confirmados a población
if (
  country_a$confirmed.population.ratio >
  country_b$confirmed.population.ratio
) {
  print(country_a$country)
} else {
  print(country_b$country)
}

In [ ]:
# 1.10 — Seleccionar los países con una relación de confirmados a población menor al 1%
threshold <- 1

Covid19_data_frame_csv[
  Covid19_data_frame_csv$confirmed.population.ratio < threshold,
  "country"
]